In [1]:
"""
Solving Ax = b Using LU (or LUP) Decomposition

We want to solve

    Ax = b

where
          [1 2 0]
    A  =  [3 4 4]
          [5 6 3]

and
          [3]
    b  =  [7]
          [8]

--------------------------------------------------
Why don't we solve Ax = b directly?
--------------------------------------------------

In the original system, every equation contains several unknowns:

    x1 + 2x2        = 3
    3x1 + 4x2 + 4x3 = 7
    5x1 + 6x2 + 3x3 = 8

All variables are mixed together.

LU decomposition factors A into

    A = LU

where

    L = lower triangular
    U = upper triangular

Triangular systems are much easier to solve because each row
contains only variables that have already been computed.

The expensive part is computing LU once (O(n^3)).

After that, solving for different b vectors only requires
forward and backward substitution (O(n^2)).

--------------------------------------------------
Step 1: Compute LU
--------------------------------------------------

Start with

          [1 2 0]
    A  =  [3 4 4]
          [5 6 3]

Use row 1 as the first pivot.

Multiplier for row 2:

    l21 = 3

Multiplier for row 3:

    l31 = 5

Perform elimination:

    R2 <- R2 - 3R1

    [3 4 4] - 3[1 2 0]
    [0 -2 4]

    R3 <- R3 - 5R1

    [5 6 3] - 5[1 2 0]
    [0 -4 3]

Matrix becomes

    [1  2  0]
    [0 -2  4]
    [0 -4  3]

Now eliminate below the second pivot.

Multiplier:

    l32 = (-4)/(-2) = 2

Perform

    R3 <- R3 - 2R2

    [0 -4 3] - 2[0 -2 4]
    [0  0 -5]

Therefore

          [1  2  0]
    U  =  [0 -2  4]
          [0  0 -5]

The multipliers form L:

          [1 0 0]
    L  =  [3 1 0]
          [5 2 1]

Verify:

    LU = A

--------------------------------------------------
Step 2: Solve Ly = b
--------------------------------------------------

We solve

          [1 0 0] [y1]   [3]
          [3 1 0] [y2] = [7]
          [5 2 1] [y3]   [8]

Row 1:

    y1 = 3

Row 2:

    3y1 + y2 = 7

    3(3) + y2 = 7

    y2 = -2

Row 3:

    5y1 + 2y2 + y3 = 8

    5(3) + 2(-2) + y3 = 8

    15 - 4 + y3 = 8

    y3 = -3

Therefore

          [ 3]
    y  =  [-2]
          [-3]

This process is called FORWARD SUBSTITUTION.

Why "forward"?

Because we solve

    y1 first,
    then y2,
    then y3.

We move from the top row downward.

--------------------------------------------------
Step 3: Solve Ux = y
--------------------------------------------------

Now solve

          [1  2  0] [x1]   [ 3]
          [0 -2  4] [x2] = [-2]
          [0  0 -5] [x3]   [-3]

Start from the bottom row.

Row 3:

    -5x3 = -3

    x3 = 3/5

Row 2:

    -2x2 + 4x3 = -2

    -2x2 + 4(3/5) = -2

    -2x2 + 12/5 = -2

    -2x2 = -22/5

    x2 = 11/5

Row 1:

    x1 + 2x2 = 3

    x1 + 2(11/5) = 3

    x1 + 22/5 = 15/5

    x1 = -7/5

Therefore

          [-7/5]
    x  =  [11/5]
          [ 3/5]

This process is called BACKWARD SUBSTITUTION.

Why "backward"?

Because we solve

    x3 first,
    then x2,
    then x1.

We move from the bottom row upward.

--------------------------------------------------
Why does LU solving work?
--------------------------------------------------

Since

    A = LU

the original system

    Ax = b

becomes

    LUx = b

Define

    y = Ux

Then

    Ly = b

which is easy to solve by forward substitution.

After finding y, solve

    Ux = y

by backward substitution.

Instead of solving one complicated system, we solve two
simple triangular systems.
"""

'\nSolving Ax = b Using LU (or LUP) Decomposition\n\nWe want to solve\n\n    Ax = b\n\nwhere\n          [1 2 0]\n    A  =  [3 4 4]\n          [5 6 3]\n\nand\n          [3]\n    b  =  [7]\n          [8]\n\n--------------------------------------------------\nWhy don\'t we solve Ax = b directly?\n--------------------------------------------------\n\nIn the original system, every equation contains several unknowns:\n\n    x1 + 2x2        = 3\n    3x1 + 4x2 + 4x3 = 7\n    5x1 + 6x2 + 3x3 = 8\n\nAll variables are mixed together.\n\nLU decomposition factors A into\n\n    A = LU\n\nwhere\n\n    L = lower triangular\n    U = upper triangular\n\nTriangular systems are much easier to solve because each row\ncontains only variables that have already been computed.\n\nThe expensive part is computing LU once (O(n^3)).\n\nAfter that, solving for different b vectors only requires\nforward and backward substitution (O(n^2)).\n\n--------------------------------------------------\nStep 1: Compute LU\n---

In [2]:
"""
A useful intuition is that plain LU and LUP are solving the same problem from different starting points.
Plain LU starts with row 1 as the first pivot. LUP says, "Before eliminating, let's move the best pivot to the top."
The permutation matrix P keeps track of those swaps so the mathematics stays equivalent. The final x is identical either way.
"""

'\nA useful intuition is that plain LU and LUP are solving the same problem from different starting points.\nPlain LU starts with row 1 as the first pivot. LUP says, "Before eliminating, let\'s move the best pivot to the top."\nThe permutation matrix P keeps track of those swaps so the mathematics stays equivalent. The final x is identical either way.\n'

In [ ]:
"""
There is not a unique LU decomposition.

One approach is to perform Gaussian elimination directly on A.

Another approach is to first swap rows so that the largest
available pivot is used. This is called partial pivoting.

With pivoting, we compute

    PA = LU

instead of

    A = LU

where

P = permutation matrix
L = lower triangular matrix
U = upper triangular matrix

Both methods solve the same system Ax = b and produce the
same final solution x.

The pivoted method is usually preferred because it is more
numerically stable and avoids dividing by very small numbers.

--------------------------------------------------
Step 1: Build the permutation matrix P
--------------------------------------------------

Look at the first column of A:

    [1]
    [3]
    [5]

The largest value is 5, which is in row 3.

Swap row 1 and row 3.

The permutation matrix is

          [0 0 1]
P =       [1 0 0]
          [0 1 0]

Multiplying P by A rearranges the rows:

           [5 6 3]
PA =       [1 2 0]
           [3 4 4]

--------------------------------------------------
Step 2: Eliminate below the first pivot
--------------------------------------------------

Pivot:

    5

--------------------------------------------------
Row 2 multiplier
--------------------------------------------------

The entry below the pivot is 1.

To eliminate it:

    multiplier = 1/5 = 0.2

Store:

    l21 = 0.2

Perform

    R2 <- R2 - 0.2 R1

    [1 2 0]
  - 0.2[5 6 3]

    [1 2 0]
  - [1 1.2 0.6]

    [0 0.8 -0.6]

--------------------------------------------------
Row 3 multiplier
--------------------------------------------------

The entry below the pivot is 3.

To eliminate it:

    multiplier = 3/5 = 0.6

Store:

    l31 = 0.6

Perform

    R3 <- R3 - 0.6 R1

    [3 4 4]
  - 0.6[5 6 3]

    [3 4 4]
  - [3 3.6 1.8]

    [0 0.4 2.2]

Current matrix:

    [5 6   3  ]
    [0 0.8 -0.6]
    [0 0.4 2.2]

--------------------------------------------------
Step 3: Eliminate below the second pivot
--------------------------------------------------

Pivot:

    0.8

Entry to eliminate:

    0.4

Multiplier:

    0.4 / 0.8 = 0.5

Store:

    l32 = 0.5

Perform

    R3 <- R3 - 0.5 R2

    [0 0.4 2.2]
  - 0.5[0 0.8 -0.6]

    [0 0.4 2.2]
  - [0 0.4 -0.3]

    [0 0 2.5]

--------------------------------------------------
Step 4: Construct L and U
--------------------------------------------------

The multipliers become the entries of L:

          [1   0   0]
L =       [0.2 1   0]
          [0.6 0.5 1]

The final matrix is U:

          [5 6   3  ]
U =       [0 0.8 -0.6]
          [0 0   2.5]

Verify:

    PA = LU

--------------------------------------------------
Step 5: Solve Ly = Pb
--------------------------------------------------

First compute Pb:

          [0 0 1]   [3]
Pb =      [1 0 0] * [7]
          [0 1 0]   [8]

          [8]
Pb =      [3]
          [7]

Now solve

          [1   0   0 ] [y1]   [8]
          [0.2 1   0 ] [y2] = [3]
          [0.6 0.5 1 ] [y3]   [7]

Row 1:

    y1 = 8

Row 2:

    0.2(8) + y2 = 3

    1.6 + y2 = 3

    y2 = 1.4

Row 3:

    0.6(8) + 0.5(1.4) + y3 = 7

    4.8 + 0.7 + y3 = 7

    y3 = 1.5

Therefore

          [8  ]
y =       [1.4]
          [1.5]

This is called FORWARD SUBSTITUTION.

--------------------------------------------------
Step 6: Solve Ux = y
--------------------------------------------------

          [5 6   3  ] [x1]   [8  ]
          [0 0.8 -0.6] [x2] = [1.4]
          [0 0   2.5] [x3]   [1.5]

Start from the bottom.

Row 3:

    2.5x3 = 1.5

    x3 = 0.6

Row 2:

    0.8x2 - 0.6(0.6) = 1.4

    0.8x2 - 0.36 = 1.4

    0.8x2 = 1.76

    x2 = 2.2

Row 1:

    5x1 + 6(2.2) + 3(0.6) = 8

    5x1 + 13.2 + 1.8 = 8

    5x1 = -7

    x1 = -1.4

Therefore

          [-1.4]
x =       [ 2.2]
          [ 0.6]

or

          [-7/5]
x =       [11/5]
          [ 3/5]

--------------------------------------------------
Why does this method still work?
--------------------------------------------------

Originally we want to solve

    Ax = b

The permutation matrix only rearranges rows.

Multiplying both sides by P gives

    PAx = Pb

Since

    PA = LU

we get

    LUx = Pb

Let

    y = Ux

Then

    Ly = Pb

Solve for y using forward substitution.

Then solve

    Ux = y

using backward substitution.

The row swaps do not change the solution x.
They only change the order of the equations.

The permutation matrix records those row swaps so that
the algebra remains correct.
"""

In [3]:
def lup_solve(L, U, pi, b):
    n = len(b)

    y = [0.0] * n
    x = [0.0] * n

    # Forward substitution: Ly = Pb
    for i in range(n):
        y[i] = b[pi[i]]

        for j in range(i):
            y[i] -= L[i][j] * y[j]

    # Backward substitution: Ux = y
    for i in range(n - 1, -1, -1):
        x[i] = y[i]

        for j in range(i + 1, n):
            x[i] -= U[i][j] * x[j]

        x[i] /= U[i][i]

    return x

In [4]:
"""
The Schur complement is not a different answer and not a different decomposition.
It's a different way of looking at the same elimination process.

In ordinary Gaussian elimination, you think:
"I'm eliminating one row at a time."

With the Schur complement viewpoint, you think:
"I'm eliminating one block of variables and updating the remaining block."

The numbers are identical, only the perspective changes.
"""

'\nThe Schur complement is not a different answer and not a different decomposition.\nIt\'s a different way of looking at the same elimination process.\n\nIn ordinary Gaussian elimination, you think:\n"I\'m eliminating one row at a time."\n\nWith the Schur complement viewpoint, you think:\n"I\'m eliminating one block of variables and updating the remaining block."\n\nThe numbers are identical, only the perspective changes.\n'

In [5]:
"""
# Schur Complement Intuition

The Schur complement is not a different answer.

It is a different way of describing the same elimination
that occurs during LU decomposition.

--------------------------------------------------
Ordinary Gaussian Elimination View
--------------------------------------------------
Suppose
          [5 6 3]
PA =      [1 2 0]
          [3 4 4]

We use the pivot 5.

Multiplier for row 2:
    1/5 = 0.2

Multiplier for row 3:
    3/5 = 0.6

Perform row operations:
    R2 <- R2 - 0.2 R1
    R3 <- R3 - 0.6 R1

Result:
    [5 6   3  ]
    [0 0.8 -0.6]
    [0 0.4  2.2]

This is the usual Gaussian elimination viewpoint.

--------------------------------------------------
Schur Complement View
--------------------------------------------------

Instead of focusing on rows, split the matrix into blocks.

          [ A11   A12 ]
PA =      [ A21   A22 ]

where

A11 = [5]
A12 = [6 3]
A21 = [1]
      [3]
A22 = [2 0]
      [4 4]

The Schur complement is
    S = A22 - A21 A11^(-1) A12

--------------------------------------------------
Compute the pieces
--------------------------------------------------

First:
    A11^(-1) = 1/5

Next:

          [1]
A21A11^(-1) =
          [3] * (1/5)

          [0.2]
        = [0.6]

Notice these are exactly the first-column multipliers
that appear in L.

Now multiply
          [0.2]
          [0.6]

by

    [6 3]

Result:
    [1.2 0.6]
    [3.6 1.8]

Now subtract from A22:

    [2 0]     [1.2 0.6]
    [4 4]  -  [3.6 1.8]

    [0.8 -0.6]
    [0.4  2.2]

--------------------------------------------------
The Key Observation
--------------------------------------------------

The resulting matrix

    [0.8 -0.6]
    [0.4  2.2]

is exactly the lower-right block obtained after the
first elimination step.

So the Schur complement is simply:

    "the remaining matrix after eliminating the first
     pivot block."

--------------------------------------------------
Why Use Blocks?
--------------------------------------------------

For a small 3x3 matrix, the row-by-row method is easier.

For a huge matrix, it is often easier to think in blocks.

Instead of eliminating one row at a time:

    eliminate a block

then

    update the remaining block

using a Schur complement.

--------------------------------------------------
Relationship to LU
--------------------------------------------------

Gaussian elimination says:

    eliminate rows

Schur complement says:

    update the remaining block

Both describe exactly the same computation.

The values in L, U, and the final solution x are
identical.

Only the viewpoint changes.

--------------------------------------------------
Simple Memory Trick
--------------------------------------------------

LU viewpoint:

    "What row operations do I perform?"

Schur complement viewpoint:

    "After eliminating a block, what smaller matrix
     remains to be solved?"

The smaller remaining matrix is the Schur complement.
"""

'\n# Schur Complement Intuition\n\nThe Schur complement is not a different answer.\n\nIt is a different way of describing the same elimination\nthat occurs during LU decomposition.\n\n--------------------------------------------------\nOrdinary Gaussian Elimination View\n--------------------------------------------------\nSuppose\n          [5 6 3]\nPA =      [1 2 0]\n          [3 4 4]\n\nWe use the pivot 5.\n\nMultiplier for row 2:\n    1/5 = 0.2\n\nMultiplier for row 3:\n    3/5 = 0.6\n\nPerform row operations:\n    R2 <- R2 - 0.2 R1\n    R3 <- R3 - 0.6 R1\n\nResult:\n    [5 6   3  ]\n    [0 0.8 -0.6]\n    [0 0.4  2.2]\n\nThis is the usual Gaussian elimination viewpoint.\n\n--------------------------------------------------\nSchur Complement View\n--------------------------------------------------\n\nInstead of focusing on rows, split the matrix into blocks.\n\n          [ A11   A12 ]\nPA =      [ A21   A22 ]\n\nwhere\n\nA11 = [5]\nA12 = [6 3]\nA21 = [1]\n      [3]\nA22 = [2 0]\n    

In [6]:
"""
Extract one row of U.
Extract one column of L.
Replace the remaining submatrix with its Schur complement.
Repeat on the smaller submatrix.
"""
def lu_decomposition(A):
    n = len(A)

    # Create L and U
    L = [[0.0] * n for _ in range(n)]
    U = [[0.0] * n for _ in range(n)]

    # Initialize L diagonal to 1
    for i in range(n):
        L[i][i] = 1.0

    # Make a working copy so we don't destroy the original
    A = [row[:] for row in A]

    for k in range(n):
        # Pivot
        U[k][k] = A[k][k]

        # Compute kth column of L and kth row of U
        for i in range(k + 1, n):
            L[i][k] = A[i][k] / A[k][k]
            U[k][i] = A[k][i]

        # Compute Schur complement
        for i in range(k + 1, n):
            for j in range(k + 1, n):
                A[i][j] = A[i][j] - L[i][k] * U[k][j]

    return L, U

In [7]:
A = [
    [5, 6, 3],
    [1, 2, 0],
    [3, 4, 4]
]

L, U = lu_decomposition(A)

print("L:")
for row in L:
    print(row)

print("\nU:")
for row in U:
    print(row)

L:
[1.0, 0.0, 0.0]
[0.2, 1.0, 0.0]
[0.6, 0.5000000000000006, 1.0]

U:
[5, 6, 3]
[0.0, 0.7999999999999998, -0.6000000000000001]
[0.0, 0.0, 2.5000000000000004]


In [1]:
"""
A modern LU algorithm really looks like:

Choose a pivot (possibly swap rows).
Form the multipliers for L.
Compute the Schur complement.
Repeat on the smaller Schur complement.

So the workflow is:

Choose pivot
      ↓
Compute multipliers
      ↓
Update Schur complement
      ↓
Repeat

The Schur complement is the updated remaining matrix.
Pivoting is the strategy for choosing a safe pivot before the update.

LU decomposition = factor the matrix.
Schur complement = the remaining subproblem after elimination.
Pivoting (PA or QA) = rearrange rows/columns so elimination is safe and numerically stable.

Pivoting (Q or P) chooses a safe pivot.
The multipliers form L.
The remaining block is the Schur complement.
LU decomposition is just repeating this block factorization recursively.
"""

'\nA modern LU algorithm really looks like:\n\nChoose a pivot (possibly swap rows).\nForm the multipliers for L.\nCompute the Schur complement.\nRepeat on the smaller Schur complement.\n\nSo the workflow is:\n\nChoose pivot\n      ↓\nCompute multipliers\n      ↓\nUpdate Schur complement\n      ↓\nRepeat\n\nThe Schur complement is the updated remaining matrix.\nPivoting is the strategy for choosing a safe pivot before the update.\n\nLU decomposition = factor the matrix.\nSchur complement = the remaining subproblem after elimination.\nPivoting (PA or QA) = rearrange rows/columns so elimination is safe and numerically stable.\n\nPivoting (Q or P) chooses a safe pivot.\nThe multipliers form L.\nThe remaining block is the Schur complement.\nLU decomposition is just repeating this block factorization recursively.\n'

In [2]:
"""
Lower triangle of A contains the multipliers (L)
Upper triangle of A contains U
pi contains the permutation
"""
def lup_decomposition(A):
    n = len(A)
    A = [row[:] for row in A]
    # Permutation vector
    pi = list(range(n))

    for k in range(n):
        # Find pivot
        p = 0.0
        pivot_row = k

        for i in range(k, n):
            if abs(A[i][k]) > p:
                p = abs(A[i][k])
                pivot_row = i

        if p == 0:
            raise ValueError("Singular matrix")

        # Swap permutation entries
        pi[k], pi[pivot_row] = pi[pivot_row], pi[k]

        # Swap rows
        A[k], A[pivot_row] = A[pivot_row], A[k]

        # Elimination
        for i in range(k + 1, n):

            A[i][k] = A[i][k] / A[k][k]

            for j in range(k + 1, n):
                A[i][j] -= A[i][k] * A[k][j]

    return A, pi

In [3]:
"""
Why Store the Permutation Vector pi Instead of the Permutation Matrix P?
During LUP decomposition, rows are swapped to place a good pivot in the current pivot position.
These row swaps are represented mathematically by a permutation matrix P.

For example, suppose the decomposition performs row swaps that result in:
    pi = [2, 0, 1]

This means:
    current row 0 came from original row 2
    current row 1 came from original row 0
    current row 2 came from original row 1

In other words, after all row swaps, the rows appear in the order:
    original row 2
    original row 0
    original row 1

The equivalent permutation matrix is
    [0 0 1]
P = [1 0 0]
    [0 1 0]

Multiplying P by A rearranges the rows of A into this order.
Instead of storing the entire matrix P, numerical algorithms store only the permutation vector pi.
This is much more efficient.

For an n×n matrix:
    P requires n² storage locations

while
    pi requires only n storage locations

For example, if n = 1000:
    P requires 1,000,000 entries

while
    pi requires only 1000 entries

This saves a large amount of memory and is one reason why practical linear algebra libraries store permutations as vectors rather than full matrices.

A useful way to remember the meaning of pi is:

    pi[i] answers the question:
        "Which original row is currently sitting in row i?"

For the example
    pi = [2, 0, 1]

the answers are:
    row 0 contains original row 2
    row 1 contains original row 0
    row 2 contains original row 1

The permutation matrix P can always be reconstructed from pi if needed,
but storing pi is much cheaper and contains exactly the same information.

Think of P as:
The mathematical object.

Think of pi as:
The computer-science implementation of that object.
"""

'\nWhy Store the Permutation Vector pi Instead of the Permutation Matrix P?\nDuring LUP decomposition, rows are swapped to place a good pivot in the current pivot position.\nThese row swaps are represented mathematically by a permutation matrix P.\n\nFor example, suppose the decomposition performs row swaps that result in:\n    pi = [2, 0, 1]\n\nThis means:\n    current row 0 came from original row 2\n    current row 1 came from original row 0\n    current row 2 came from original row 1\n\nIn other words, after all row swaps, the rows appear in the order:\n    original row 2\n    original row 0\n    original row 1\n\nThe equivalent permutation matrix is\n    [0 0 1]\nP = [1 0 0]\n    [0 1 0]\n\nMultiplying P by A rearranges the rows of A into this order.\nInstead of storing the entire matrix P, numerical algorithms store only the permutation vector pi.\nThis is much more efficient.\n\nFor an n×n matrix:\n    P requires n² storage locations\n\nwhile\n    pi requires only n storage locat

In [ ]:
"""
Small forward substitution example:

[ 1  0  0 ] [x1]   [ 3 ]
[ 4  1  0 ] [x2] = [14 ]
[-6  5  1 ] [x3]   [-7 ]

Forward substitution:

Row 1:
x1 = 3

Row 2:
4x1 + x2 = 14
4(3) + x2 = 14
12 + x2 = 14
x2 = 2

Row 3:
-6x1 + 5x2 + x3 = -7
-6(3) + 5(2) + x3 = -7
-18 + 10 + x3 = -7
-8 + x3 = -7
x3 = 1

Solution:
[x1, x2, x3] = [3, 2, 1]
============================================
============================================
============================================
Small LU example:

A =
[ 4  -5   6 ]
[ 8  -6   7 ]
[12  -7  12 ]

Pivot a11 = 4

l21 = 8/4 = 2
l31 = 12/4 = 3

R2 <- R2 - 2R1
[8 -6 7] - 2[4 -5 6]
= [0 4 -5]

R3 <- R3 - 3R1
[12 -7 12] - 3[4 -5 6]
= [0 8 -6]

Matrix becomes
[4 -5  6 ]
[0  4 -5 ]
[0  8 -6 ]

Pivot a22 = 4
l32 = 8/4 = 2

R3 <- R3 - 2R2
[0 8 -6] - 2[0 4 -5]
= [0 0 4]

U =
[4 -5  6 ]
[0  4 -5 ]
[0  0  4 ]

L =
[1 0 0]
[2 1 0]
[3 2 1]

Therefore A = LU
============================================
============================================
============================================
Small LUP Example:
Solve Ax = b using LUP Decomposition

Given

A =
[1  5  4]
[2  0  3]
[5  8  2]

b =
[12]
[ 9]
[ 5]

We want to solve
Ax = b

using LUP decomposition.
--------------------------------------------------
Step 1: Pivot in Column 1
--------------------------------------------------

Column 1 is
[1]
[2]
[5]

The largest absolute value is 5 (row 3).
Swap row 1 and row 3.

P =
[0 0 1]
[0 1 0]
[1 0 0]

PA =
[5 8 2]
[2 0 3]
[1 5 4]

--------------------------------------------------
Step 2: Eliminate Below the First Pivot
--------------------------------------------------

Pivot:
5

Multipliers:
l21 = 2/5
l31 = 1/5

Store these in L.

--------------------------------------------------
Row 2 Elimination
--------------------------------------------------

R2 <- R2 - (2/5)R1
[2 0 3] - (2/5)[5 8 2]

= [2 0 3] - [2 16/5 4/5]
= [0 -8/5 11/5]

--------------------------------------------------
Row 3 Elimination
--------------------------------------------------

R3 <- R3 - (1/5)R1

[1 5 4] - (1/5)[5 8 2]

= [1 5 4] - [1 8/5 2/5]
= [0 17/5 18/5]

Matrix becomes
[5   8    2   ]
[0  -8/5 11/5]
[0  17/5 18/5]

--------------------------------------------------
Step 3: Pivot in Column 2
--------------------------------------------------

Compare

|-8/5| = 1.6
|17/5| = 3.4

Largest absolute value is 17/5.
Swap rows 2 and 3.

Matrix becomes
[5   8    2   ]
[0  17/5 18/5]
[0  -8/5 11/5]

Permutation matrix becomes

P =
[0 0 1]
[1 0 0]
[0 1 0]

--------------------------------------------------
Step 4: Eliminate Below the Second Pivot
--------------------------------------------------

Pivot:
17/5

Multiplier:
l32 = (-8/5)/(17/5)
    = -8/17

Store this in L.

--------------------------------------------------
Row 3 Elimination
--------------------------------------------------

R3 <- R3 - (-8/17)R2

Third entry:

11/5 - (-8/17)(18/5)

= 11/5 + 144/85
= 331/85

Thus

U =
[5   8    2     ]
[0  17/5 18/5   ]
[0   0   331/85 ]

--------------------------------------------------
Step 5: Construct L
--------------------------------------------------

The multipliers are

l21 = 1/5
l31 = 2/5
l32 = -8/17

Therefore

L =
[1    0      0]
[1/5  1      0]
[2/5 -8/17   1]

--------------------------------------------------
Step 6: Compute Pb
--------------------------------------------------

P =
[0 0 1]
[1 0 0]
[0 1 0]

b =
[12]
[ 9]
[ 5]

Pb =
[ 5]
[12]
[ 9]

--------------------------------------------------
Step 7: Solve Ly = Pb
--------------------------------------------------

[1    0      0] [y1]   [ 5]
[1/5  1      0] [y2] = [12]
[2/5 -8/17   1] [y3]   [ 9]

Row 1:
y1 = 5

Row 2:
(1/5)(5) + y2 = 12

1 + y2 = 12
y2 = 11

Row 3:
(2/5)(5) - (8/17)(11) + y3 = 9
2 - 88/17 + y3 = 9
y3 = 207/17

Therefore

y =
[  5   ]
[ 11   ]
[207/17]

--------------------------------------------------
Step 8: Solve Ux = y
--------------------------------------------------

[5   8    2     ] [x1]   [  5   ]
[0  17/5 18/5   ] [x2] = [ 11   ]
[0   0   331/85 ] [x3]   [207/17]

Row 3:
(331/85)x3 = 207/17
x3 = 45/17

Row 2:
(17/5)x2 + (18/5)(45/17) = 11
17x2 + 810/17 = 55
289x2 = 125
x2 = 125/289

Row 1:
5x1 + 8(125/289) + 2(45/17) = 5
5x1 + 1000/289 + 90/17 = 5
5x1 = -1085/289
x1 = -217/289

--------------------------------------------------
Solution
--------------------------------------------------

x =
[-217/289]
[ 125/289]
[  45/17 ]

Approximate values:

x ≈
[-0.7509]
[ 0.4325]
[ 2.6471]

Verification:
Ax = b

Therefore the solution is
x = [-217/289, 125/289, 45/17]
============================================
============================================
============================================
For a nonsingular diagonal matrix

Example of a diagonal matrix:

A =
[4 0 0]
[0 7 0]
[0 0 2]

Notice that all entries off the main diagonal are zero.

The main diagonal entries are:
    4, 7, 2
Since there are already zeros below the diagonal, no elimination is needed.

P =
[1 0 0]
[0 1 0]
[0 0 1]

L =
[1 0 0]
[0 1 0]
[0 0 1]

U =
[4 0 0]
[0 7 0]
[0 0 2]

Verification:

PA = LU
[1 0 0]   [4 0 0]
[0 1 0] * [0 7 0]
[0 0 1]   [0 0 2]

=
[4 0 0]
[0 7 0]
[0 0 2]

and

[1 0 0]   [4 0 0]
[0 1 0] * [0 7 0]
[0 0 1]   [0 0 2]

=
[4 0 0]
[0 7 0]
[0 0 2]

Therefore:

P = I
L = I
U = A

for any nonsingular diagonal matrix.
"""

In [4]:
"""
The only permutation matrix with an LU decomposition is I.
A =
[0 0 1]
[1 0 0]
[0 1 0]

Then

P = A^T =
[0 1 0]
[0 0 1]
[1 0 0]

Multiplying:

PA =
[0 1 0]   [0 0 1]
[0 0 1] * [1 0 0]
[1 0 0]   [0 1 0]

=
[1 0 0]
[0 1 0]
[0 0 1]
= I

Hence
L = I
U = I.

"""

'\nThe only permutation matrix with an LU decomposition is I.\nA =\n[0 0 1]\n[1 0 0]\n[0 1 0]\n\nThen\n\nP = A^T =\n[0 1 0]\n[0 0 1]\n[1 0 0]\n\nMultiplying:\n\nPA =\n[0 1 0]   [0 0 1]\n[0 0 1] * [1 0 0]\n[1 0 0]   [0 1 0]\n\n=\n[1 0 0]\n[0 1 0]\n[0 0 1]\n= I\n\nHence\nL = I\nU = I.\n\n'

In [6]:
"""
An invertible matrix:
stretches,
rotates,
reflects,
but does not collapse dimensions.

A singular matrix:
collapses space
maps every point onto a single line
Once a whole plane is squashed onto a line, you can't uniquely recover the original point, so there can be no inverse.

Invertible:
    determinant ≠ 0
    inverse exists
    Ax = b has a unique solution for every b

Singular:
    determinant = 0
    inverse does not exist
    Ax = b may have no solution or infinitely many solutions

Having an LU decomposition does NOT require a matrix to be invertible.


Consider the matrix

A =
[1 0 0 ... 0]
[0 1 0 ... 0]
[0 0 1 ... 0]
[. . .     .]
[0 0 0 ... 0]

which is the identity matrix except that the last diagonal
entry is replaced by 0.

For example, when n = 4,

A =
[1 0 0 0]
[0 1 0 0]
[0 0 1 0]
[0 0 0 0]

--------------------------------------------------
Why is A singular?
--------------------------------------------------

The determinant of a triangular matrix is the product of its
diagonal entries.

Therefore

det(A) = 1 · 1 · ... · 1 · 0 = 0.

Hence A is singular.

--------------------------------------------------
Why does A have an LU decomposition?
--------------------------------------------------

Take

L = I

(the n×n identity matrix)

and

U = A.

Since A is already upper triangular,

    U = A

is a valid upper triangular matrix.

Also

    L = I

is a valid unit lower triangular matrix.

Then

LU = IA = A.

Therefore

A = LU.

For every n ≥ 1, the matrix

A = diag(1,1,...,1,0)
is singular and satisfies

A = LU

with

L = I
U = A

Hence for every n ≥ 1, there exists a singular n×n matrix
that has an LU decomposition.
"""

"\nAn invertible matrix:\nstretches,\nrotates,\nreflects,\nbut does not collapse dimensions.\n\nA singular matrix:\ncollapses space\nmaps every point onto a single line\nOnce a whole plane is squashed onto a line, you can't uniquely recover the original point, so there can be no inverse.\n\nInvertible:\n    determinant ≠ 0\n    inverse exists\n    Ax = b has a unique solution for every b\n\nSingular:\n    determinant = 0\n    inverse does not exist\n    Ax = b may have no solution or infinitely many solutions\n\nHaving an LU decomposition does NOT require a matrix to be invertible.\n\n\nConsider the matrix\n\nA =\n[1 0 0 ... 0]\n[0 1 0 ... 0]\n[0 0 1 ... 0]\n[. . .     .]\n[0 0 0 ... 0]\n\nwhich is the identity matrix except that the last diagonal\nentry is replaced by 0.\n\nFor example, when n = 4,\n\nA =\n[1 0 0 0]\n[0 1 0 0]\n[0 0 1 0]\n[0 0 0 0]\n\n--------------------------------------------------\nWhy is A singular?\n--------------------------------------------------\n\nThe deter

In [7]:
"""
In LU without pivoting, you've already assumed every pivot exists.
In LUP, the final iteration performs the last singularity check.

The k = n iteration performs no elimination.

However, it still performs the final pivot search and
checks whether the last pivot is zero.

Therefore the k = n iteration is generally required in
LUP decomposition to correctly detect singular matrices.

Why the final iteration matters more in LUP
In LU without pivoting, you've already assumed every pivot exists.
In LUP, the final iteration performs the last singularity check.

LU decomposition:
    k = n performs no elimination.
    It is only needed to store the final diagonal element of U.
    It can often be omitted if handled separately.

LUP decomposition:
    k = n performs no elimination.
    However, it verifies that the final pivot is nonzero.
    Therefore it should generally be retained to correctly detect singular matrices.
"""

"\nIn LU without pivoting, you've already assumed every pivot exists.\nIn LUP, the final iteration performs the last singularity check.\n\nThe k = n iteration performs no elimination.\n\nHowever, it still performs the final pivot search and\nchecks whether the last pivot is zero.\n\nTherefore the k = n iteration is generally required in\nLUP decomposition to correctly detect singular matrices.\n\nWhy the final iteration matters more in LUP\nIn LU without pivoting, you've already assumed every pivot exists.\nIn LUP, the final iteration performs the last singularity check.\n\nLU decomposition:\n    k = n performs no elimination.\n    It is only needed to store the final diagonal element of U.\n    It can often be omitted if handled separately.\n\nLUP decomposition:\n    k = n performs no elimination.\n    However, it verifies that the final pivot is nonzero.\n    Therefore it should generally be retained to correctly detect singular matrices.\n"

In [9]:
"""
1. Multiplication -> Squaring
   A^2 = A*A

   One matrix multiplication computes a square.

   Therefore
       S(n) <= M(n)

   and
       S(n) = O(M(n))

--------------------------------------------------
2. Squaring -> Multiplication

   Given A and B, construct
       X = [0 A]
           [B 0]

   Then
       X^2 = [AB  0]
             [0  BA]

   The product AB appears in the upper-left block.

   Therefore one squaring algorithm computes
   matrix multiplication.

   Thus
       M(n) = O(S(n))

--------------------------------------------------

Conclusion
       M(n) = Θ(S(n))

Matrix multiplication and matrix squaring have
essentially the same computational difficulty.
"""

'\n1. Multiplication -> Squaring\n   A^2 = A*A\n\n   One matrix multiplication computes a square.\n\n   Therefore\n       S(n) <= M(n)\n\n   and\n       S(n) = O(M(n))\n\n--------------------------------------------------\n2. Squaring -> Multiplication\n\n   Given A and B, construct\n       X = [0 A]\n           [B 0]\n\n   Then\n       X^2 = [AB  0]\n             [0  BA]\n\n   The product AB appears in the upper-left block.\n\n   Therefore one squaring algorithm computes\n   matrix multiplication.\n\n   Thus\n       M(n) = O(S(n))\n\n--------------------------------------------------\n\nConclusion\n       M(n) = Θ(S(n))\n\nMatrix multiplication and matrix squaring have\nessentially the same computational difficulty.\n'

In [10]:
"""
Assume an M(n)-time algorithm for multiplying n×n matrices.

Use a recursive block LUP decomposition.

Partition

    A = [A11 A12]
        [A21 A22]

Recursively compute the LUP decomposition of A11.

Then form the Schur complement

    S = A22 - A21 A11^(-1) A12.

The dominant work in forming S is matrix multiplication,
which takes O(M(n)) time by assumption.

Recursively compute the LUP decomposition of S.

This yields the recurrence

    T(n) = 2T(n/2) + O(M(n)).

Since M(n) = Ω(n²), the O(M(n)) term dominates the
recursion tree, giving

    T(n) = O(M(n)).

Therefore an M(n)-time matrix multiplication algorithm
implies an O(M(n))-time LUP decomposition algorithm.
"""

'\nAssume an M(n)-time algorithm for multiplying n×n matrices.\n\nUse a recursive block LUP decomposition.\n\nPartition\n\n    A = [A11 A12]\n        [A21 A22]\n\nRecursively compute the LUP decomposition of A11.\n\nThen form the Schur complement\n\n    S = A22 - A21 A11^(-1) A12.\n\nThe dominant work in forming S is matrix multiplication,\nwhich takes O(M(n)) time by assumption.\n\nRecursively compute the LUP decomposition of S.\n\nThis yields the recurrence\n\n    T(n) = 2T(n/2) + O(M(n)).\n\nSince M(n) = Ω(n²), the O(M(n)) term dominates the\nrecursion tree, giving\n\n    T(n) = O(M(n)).\n\nTherefore an M(n)-time matrix multiplication algorithm\nimplies an O(M(n))-time LUP decomposition algorithm.\n'

In [14]:
"""
Multiplication → Closure: use repeated squaring of I∨A.
Closure → Multiplication: use the three-layer graph/block matrix
    [0 A 0]
X = [0 0 B]
    [0 0 0]

Boolean matrix multiplication and transitive closure
are computationally equivalent up to a logarithmic
factor: T(n) = O(M(n) log n)

and M(n) = O(T(n))
"""

'\nMultiplication → Closure: use repeated squaring of I∨A.\nClosure → Multiplication: use the three-layer graph/block matrix\n    [0 A 0]\nX = [0 0 B]\n    [0 0 0]\n\nBoolean matrix multiplication and transitive closure\nare computationally equivalent up to a logarithmic\nfactor: T(n) = O(M(n) log n)\n\nand M(n) = O(T(n))\n'

In [13]:
"""
Mod 2 syntax/matrix notation:
Over R:
    Ri <- Ri + Rj
means ordinary addition.

Over F2:
    Ri <- Ri + Rj
means addition modulo 2.

Mathematics:
    Ri <- Ri + Rj

Explicit field arithmetic:
    Ri <- (Ri + Rj) mod 2

Programming:
    Ri <- Ri XOR Rj
"""

'\nMod 2 syntax/matrix notation:\nOver R:\n    Ri <- Ri + Rj\nmeans ordinary addition.\n\nOver F2:\n    Ri <- Ri + Rj\nmeans addition modulo 2.\n\nMathematics:\n    Ri <- Ri + Rj\n\nExplicit field arithmetic:\n    Ri <- (Ri + Rj) mod 2\n\nProgramming:\n    Ri <- Ri XOR Rj\n'

In [16]:
"""
To generalize the matrix-inversion algorithm from real matrices
to complex matrices, replace every occurrence of the transpose
A^T by the conjugate transpose A*.

For a complex matrix A = (aij),
    A* = (conjugate(aji)).

A matrix satisfying
    A = A*

is called Hermitian. Hermitian matrices are the complex
analogues of symmetric matrices.

The matrix-inversion algorithm still works because Gaussian
elimination requires only that matrix entries belong to a field
and that every nonzero pivot have a multiplicative inverse.
The complex numbers form a field, so all elementary row
operations remain valid.

The key identity used in the real case,
    A^T A,

is replaced by
    A* A.

For any complex vector x,

    x* A* A x
      = (Ax)* (Ax)
      = sum |(Ax)i|^2 >= 0.

Moreover,
    x* A* A x = 0

if and only if
    Ax = 0.

Therefore A* A is Hermitian positive definite whenever A is
invertible.

Since all arguments used in the correctness proof for the real
case continue to hold after replacing transpose by conjugate
transpose and symmetric matrices by Hermitian matrices, the
generalized matrix-inversion algorithm is correct for matrices
with complex entries.


Number      Matrix Concept

Real        Symmetric (A = A^T)

Complex     Hermitian (A = A*)

Absolute value:
    |z| = sqrt(z * conjugate(z))

Inverse:
    Tells whether a transformation can be undone.

Transpose:
    Swaps rows and columns.

Conjugate transpose:
    Swaps rows/columns and flips i -> -i.

Complex matrices:
    Used in waves, signals, circuits,
    quantum mechanics, and Fourier analysis.

Real case                 Complex case

Transpose A^T      --->   Conjugate transpose A*

Symmetric          --->   Hermitian

A^T A              --->   A* A
"""

'\nTo generalize the matrix-inversion algorithm from real matrices\nto complex matrices, replace every occurrence of the transpose\nA^T by the conjugate transpose A*.\n\nFor a complex matrix A = (aij),\n    A* = (conjugate(aji)).\n\nA matrix satisfying\n    A = A*\n\nis called Hermitian. Hermitian matrices are the complex\nanalogues of symmetric matrices.\n\nThe matrix-inversion algorithm still works because Gaussian\nelimination requires only that matrix entries belong to a field\nand that every nonzero pivot have a multiplicative inverse.\nThe complex numbers form a field, so all elementary row\noperations remain valid.\n\nThe key identity used in the real case,\n    A^T A,\n\nis replaced by\n    A* A.\n\nFor any complex vector x,\n\n    x* A* A x\n      = (Ax)* (Ax)\n      = sum |(Ax)i|^2 >= 0.\n\nMoreover,\n    x* A* A x = 0\n\nif and only if\n    Ax = 0.\n\nTherefore A* A is Hermitian positive definite whenever A is\ninvertible.\n\nSince all arguments used in the correctness proof

In [17]:
"""
Transpose is the matrix version of "turning something sideways."
Inverse is the matrix version of "undoing."
Conjugate is the complex-number version of a mirror reflection across the real axis.
Hermitian matrices are what symmetric matrices become when the world includes complex numbers.
"""

'\nTranspose is the matrix version of "turning something sideways."\nInverse is the matrix version of "undoing."\nConjugate is the complex-number version of a mirror reflection across the real axis.\nHermitian matrices are what symmetric matrices become when the world includes complex numbers.\n'

In [18]:
"""
Example of finding least-squares approximation

F(x) = c₁ + c₂xln(x) + c₃xeˣ

to the data points

(1,1), (2,1), (3,3), (4,8).

Step 1: Define the basis functions.

φ₁(x) = 1
φ₂(x) = xln(x)
φ₃(x) = xeˣ

Since the model is a linear combination of these basis functions, we can write

F(x) = c₁φ₁(x) + c₂φ₂(x) + c₃φ₃(x).

Step 2: Construct the design matrix A by evaluating each basis function at the x-values.

For x = 1:

φ₁(1) = 1
φ₂(1) = 1ln(1) = 0
φ₃(1) = e ≈ 2.7183

For x = 2:

φ₁(2) = 1
φ₂(2) = 2ln(2) ≈ 1.3863
φ₃(2) = 2e² ≈ 14.7781

For x = 3:

φ₁(3) = 1
φ₂(3) = 3ln(3) ≈ 3.2958
φ₃(3) = 3e³ ≈ 60.2566

For x = 4:

φ₁(4) = 1
φ₂(4) = 4ln(4) ≈ 5.5452
φ₃(4) = 4e⁴ ≈ 218.3926

Thus,

A = [
 [1, 0,      2.7183],
 [1, 1.3863, 14.7781],
 [1, 3.2958, 60.2566],
 [1, 5.5452, 218.3926]
]

and the observation vector is

b = [1, 1, 3, 8]ᵀ.

Step 3: Form the normal equations.

The least-squares solution satisfies

AᵀAc = Aᵀb.

Compute AᵀA:

AᵀA ≈

[
 [4,       10.2273,   296.1456],
 [10.2273, 42.2769,  1415.3650],
 [296.1456,1415.3650,51646.6920]
]

Compute Aᵀb:

Aᵀb ≈

[
 [13],
 [54.2488],
 [1938.6295]
]

Therefore the normal equations are

[
 [4,       10.2273,   296.1456],
 [10.2273, 42.2769,  1415.3650],
 [296.1456,1415.3650,51646.6920]
]

[c₁, c₂, c₃]ᵀ

=

[
 [13],
 [54.2488],
 [1938.6295]
].

Step 4: Solve the linear system.

Solving for the coefficients gives

c₁ ≈ 0.779
c₂ ≈ -2.214
c₃ ≈ 0.0386

Step 5: Substitute the coefficients into the model.

F(x) = 0.779 - 2.214xln(x) + 0.0386xeˣ

Therefore, the least-squares approximation is

F(x) ≈ 0.779 - 2.214xln(x) + 0.0386xeˣ.
"""

'\nExample of finding least-squares approximation\n\nF(x) = c₁ + c₂xln(x) + c₃xeˣ\n\nto the data points\n\n(1,1), (2,1), (3,3), (4,8).\n\nStep 1: Define the basis functions.\n\nφ₁(x) = 1\nφ₂(x) = xln(x)\nφ₃(x) = xeˣ\n\nSince the model is a linear combination of these basis functions, we can write\n\nF(x) = c₁φ₁(x) + c₂φ₂(x) + c₃φ₃(x).\n\nStep 2: Construct the design matrix A by evaluating each basis function at the x-values.\n\nFor x = 1:\n\nφ₁(1) = 1\nφ₂(1) = 1ln(1) = 0\nφ₃(1) = e ≈ 2.7183\n\nFor x = 2:\n\nφ₁(2) = 1\nφ₂(2) = 2ln(2) ≈ 1.3863\nφ₃(2) = 2e² ≈ 14.7781\n\nFor x = 3:\n\nφ₁(3) = 1\nφ₂(3) = 3ln(3) ≈ 3.2958\nφ₃(3) = 3e³ ≈ 60.2566\n\nFor x = 4:\n\nφ₁(4) = 1\nφ₂(4) = 4ln(4) ≈ 5.5452\nφ₃(4) = 4e⁴ ≈ 218.3926\n\nThus,\n\nA = [\n [1, 0,      2.7183],\n [1, 1.3863, 14.7781],\n [1, 3.2958, 60.2566],\n [1, 5.5452, 218.3926]\n]\n\nand the observation vector is\n\nb = [1, 1, 3, 8]ᵀ.\n\nStep 3: Form the normal equations.\n\nThe least-squares solution satisfies\n\nAᵀAc = Aᵀb.\n\nCompute AᵀA

In [20]:
"""
many matrices are:

rectangular,
singular,
rank deficient.

For these matrices, an ordinary inverse does not exist.
The pseudoinverse A+ acts as the best possible inverse.

The Moore-Penrose pseudoinverse A⁺ is defined as the unique matrix satisfying four properties:

1. AA⁺A = A
2. A⁺AA⁺ = A⁺
3. (AA⁺)ᵀ = AA⁺
4. (A⁺A)ᵀ = A⁺A

Example:

Let

A = [[1],
     [2]]

Since A is a 2×1 matrix, it has no ordinary inverse.

Compute the pseudoinverse using

A⁺ = (AᵀA)⁻¹Aᵀ.

First,

AᵀA = [1 2][[1],[2]] = 5.

Thus,

A⁺ = (1/5)[1 2].

Property 1:

AA⁺ = [[1],[2]](1/5)[1 2]

     = [[1/5, 2/5],
        [2/5, 4/5]]

Then

AA⁺A

= [[1/5, 2/5],
   [2/5, 4/5]]
  [[1],
   [2]]

= [[1],
   [2]]

= A.

Therefore AA⁺A = A.

Property 2:

A⁺A

= (1/5)[1 2][[1],[2]]

= [1].

Therefore

A⁺AA⁺ = [1]A⁺ = A⁺.

Property 3:

AA⁺

= [[1/5, 2/5],
   [2/5, 4/5]]

Taking the transpose gives the same matrix, so

(AA⁺)ᵀ = AA⁺.

Property 4:

A⁺A = [1].

Since [1]ᵀ = [1],

(A⁺A)ᵀ = A⁺A.

Geometrically, AA⁺ projects vectors onto the column space of A, while A⁺A projects vectors onto the row space of A.
Both projection matrices are symmetric, which explains the transpose properties.
The first two properties say that the pseudoinverse behaves like an inverse whenever possible, even when A is rectangular or singular.
"""

'\nmany matrices are:\n\nrectangular,\nsingular,\nrank deficient.\n\nFor these matrices, an ordinary inverse does not exist.\nThe pseudoinverse A+ acts as the best possible inverse.\n\nThe Moore-Penrose pseudoinverse A⁺ is defined as the unique matrix satisfying four properties:\n\n1. AA⁺A = A\n2. A⁺AA⁺ = A⁺\n3. (AA⁺)ᵀ = AA⁺\n4. (A⁺A)ᵀ = A⁺A\n\nExample:\n\nLet\n\nA = [[1],\n     [2]]\n\nSince A is a 2×1 matrix, it has no ordinary inverse.\n\nCompute the pseudoinverse using\n\nA⁺ = (AᵀA)⁻¹Aᵀ.\n\nFirst,\n\nAᵀA = [1 2][[1],[2]] = 5.\n\nThus,\n\nA⁺ = (1/5)[1 2].\n\nProperty 1:\n\nAA⁺ = [[1],[2]](1/5)[1 2]\n\n     = [[1/5, 2/5],\n        [2/5, 4/5]]\n\nThen\n\nAA⁺A\n\n= [[1/5, 2/5],\n   [2/5, 4/5]]\n  [[1],\n   [2]]\n\n= [[1],\n   [2]]\n\n= A.\n\nTherefore AA⁺A = A.\n\nProperty 2:\n\nA⁺A\n\n= (1/5)[1 2][[1],[2]]\n\n= [1].\n\nTherefore\n\nA⁺AA⁺ = [1]A⁺ = A⁺.\n\nProperty 3:\n\nAA⁺\n\n= [[1/5, 2/5],\n   [2/5, 4/5]]\n\nTaking the transpose gives the same matrix, so\n\n(AA⁺)ᵀ = AA⁺.\n\nProperty 

In [21]:
"""
PART (a) - LU Decomposition

A =
[ 1  -1   0   0   0 ]
[ -1  2  -1   0   0 ]
[ 0  -1   2  -1   0 ]
[ 0   0  -1   2  -1 ]
[ 0   0   0  -1   2 ]

Perform Gaussian elimination without pivoting.

Step 1:
Pivot = a11 = 1

Multiplier:
m21 = (-1)/1 = -1

Perform
R2 <- R2 - m21*R1
R2 <- R2 + R1

[-1  2  -1  0  0] + [1 -1 0 0 0]
= [0 1 -1 0 0]

Step 2:
Pivot = a22 = 1

Multiplier:
m32 = (-1)/1 = -1

Perform
R3 <- R3 + R2

[0 -1 2 -1 0] + [0 1 -1 0 0]
= [0 0 1 -1 0]

Step 3:
Pivot = a33 = 1

Multiplier:
m43 = -1

Perform
R4 <- R4 + R3

[0 0 -1 2 -1] + [0 0 1 -1 0]
= [0 0 0 1 -1]

Step 4:
Pivot = a44 = 1

Multiplier:
m54 = -1

Perform
R5 <- R5 + R4

[0 0 0 -1 2] + [0 0 0 1 -1]
= [0 0 0 0 1]

Therefore
L =
[ 1  0  0  0  0 ]
[ -1 1  0  0  0 ]
[ 0 -1  1  0  0 ]
[ 0  0 -1  1  0 ]
[ 0  0  0 -1  1 ]

U =
[ 1 -1  0  0  0 ]
[ 0  1 -1  0  0 ]
[ 0  0  1 -1  0 ]
[ 0  0  0  1 -1 ]
[ 0  0  0  0  1 ]

Hence

A = LU
"""

'\nPART (a) - LU Decomposition\n\nA =\n[ 1  -1   0   0   0 ]\n[ -1  2  -1   0   0 ]\n[ 0  -1   2  -1   0 ]\n[ 0   0  -1   2  -1 ]\n[ 0   0   0  -1   2 ]\n\nPerform Gaussian elimination without pivoting.\n\nStep 1:\nPivot = a11 = 1\n\nMultiplier:\nm21 = (-1)/1 = -1\n\nPerform\nR2 <- R2 - m21*R1\nR2 <- R2 + R1\n\n[-1  2  -1  0  0] + [1 -1 0 0 0]\n= [0 1 -1 0 0]\n\nStep 2:\nPivot = a22 = 1\n\nMultiplier:\nm32 = (-1)/1 = -1\n\nPerform\nR3 <- R3 + R2\n\n[0 -1 2 -1 0] + [0 1 -1 0 0]\n= [0 0 1 -1 0]\n\nStep 3:\nPivot = a33 = 1\n\nMultiplier:\nm43 = -1\n\nPerform\nR4 <- R4 + R3\n\n[0 0 -1 2 -1] + [0 0 1 -1 0]\n= [0 0 0 1 -1]\n\nStep 4:\nPivot = a44 = 1\n\nMultiplier:\nm54 = -1\n\nPerform\nR5 <- R5 + R4\n\n[0 0 0 -1 2] + [0 0 0 1 -1]\n= [0 0 0 0 1]\n\nTherefore\nL =\n[ 1  0  0  0  0 ]\n[ -1 1  0  0  0 ]\n[ 0 -1  1  0  0 ]\n[ 0  0 -1  1  0 ]\n[ 0  0  0 -1  1 ]\n\nU =\n[ 1 -1  0  0  0 ]\n[ 0  1 -1  0  0 ]\n[ 0  0  1 -1  0 ]\n[ 0  0  0  1 -1 ]\n[ 0  0  0  0  1 ]\n\nHence\n\nA = LU\n'

In [22]:
"""
PART (b) - Solve Ax = b

b =
[1]
[1]
[1]
[1]
[1]

Since A = LU, solve
Ly = b

Forward substitution:

Row 1:
y1 = 1

Row 2:
-y1 + y2 = 1
-1 + y2 = 1
y2 = 2

Row 3:
-y2 + y3 = 1
-2 + y3 = 1
y3 = 3

Row 4:
-y3 + y4 = 1
-3 + y4 = 1
y4 = 4

Row 5:
-y4 + y5 = 1
-4 + y5 = 1
y5 = 5

Therefore
y =
[1]
[2]
[3]
[4]
[5]

Now solve
Ux = y

Back substitution:
Row 5:
x5 = 5

Row 4:
x4 - x5 = 4
x4 = 9

Row 3:
x3 - x4 = 3
x3 = 12

Row 2:
x2 - x3 = 2
x2 = 14

Row 1:
x1 - x2 = 1
x1 = 15
Therefore

x =
[15]
[14]
[12]
[ 9]
[ 5]
"""

'\nPART (b) - Solve Ax = b\n\nb =\n[1]\n[1]\n[1]\n[1]\n[1]\n\nSince A = LU, solve\nLy = b\n\nForward substitution:\n\nRow 1:\ny1 = 1\n\nRow 2:\n-y1 + y2 = 1\n-1 + y2 = 1\ny2 = 2\n\nRow 3:\n-y2 + y3 = 1\n-2 + y3 = 1\ny3 = 3\n\nRow 4:\n-y3 + y4 = 1\n-3 + y4 = 1\ny4 = 4\n\nRow 5:\n-y4 + y5 = 1\n-4 + y5 = 1\ny5 = 5\n\nTherefore\ny =\n[1]\n[2]\n[3]\n[4]\n[5]\n\nNow solve\nUx = y\n\nBack substitution:\nRow 5:\nx5 = 5\n\nRow 4:\nx4 - x5 = 4\nx4 = 9\n\nRow 3:\nx3 - x4 = 3\nx3 = 12\n\nRow 2:\nx2 - x3 = 2\nx2 = 14\n\nRow 1:\nx1 - x2 = 1\nx1 = 15\nTherefore\n\nx =\n[15]\n[14]\n[12]\n[ 9]\n[ 5]\n'

In [27]:
"""
PART (c) - Finding A^(-1) by Solving Ax_i = e_i

Recall that the inverse matrix is formed by solving

Ax1 = e1
Ax2 = e2
Ax3 = e3
Ax4 = e4
Ax5 = e5

where

e1 = [1 0 0 0 0]^T
e2 = [0 1 0 0 0]^T
e3 = [0 0 1 0 0]^T
e4 = [0 0 0 1 0]^T
e5 = [0 0 0 0 1]^T

The resulting vectors become the columns of A^(-1).

Since A = LU, solve each system in two steps:

1. Solve Ly = e_i
2. Solve Ux_i = y

--------------------------------------------------
COLUMN 1: Solve Ax1 = e1
--------------------------------------------------

e1 =
[1]
[0]
[0]
[0]
[0]

Step 1: Solve Ly = e1

L =
[ 1  0  0  0  0 ]
[-1  1  0  0  0 ]
[ 0 -1  1  0  0 ]
[ 0  0 -1  1  0 ]
[ 0  0  0 -1  1 ]

Row 1:
y1 = 1

Row 2:
-y1 + y2 = 0
-1 + y2 = 0
y2 = 1

Row 3:
-y2 + y3 = 0
-1 + y3 = 0
y3 = 1

Row 4:
-y3 + y4 = 0
-1 + y4 = 0
y4 = 1

Row 5:
-y4 + y5 = 0
-1 + y5 = 0
y5 = 1

Therefore

y =
[1]
[1]
[1]
[1]
[1]

Step 2: Solve Ux1 = y

U =
[1 -1  0  0  0]
[0  1 -1  0  0]
[0  0  1 -1  0]
[0  0  0  1 -1]
[0  0  0  0  1]

Row 5:
x5 = 1

Row 4:
x4 - x5 = 1
x4 - 1 = 1
x4 = 2

Row 3:
x3 - x4 = 1
x3 - 2 = 1
x3 = 3

Row 2:
x2 - x3 = 1
x2 - 3 = 1
x2 = 4

Row 1:
x1 - x2 = 1
x1 - 4 = 1
x1 = 5

Therefore
Column 1 =
x1 =
[5]
[4]
[3]
[2]
[1]

--------------------------------------------------
COLUMN 2: Solve Ax2 = e2
--------------------------------------------------

e2 =
[0]
[1]
[0]
[0]
[0]

Step 1: Solve Ly = e2

Row 1:
y1 = 0

Row 2:
-y1 + y2 = 1
0 + y2 = 1
y2 = 1

Row 3:
-y2 + y3 = 0
-1 + y3 = 0
y3 = 1

Row 4:
-y3 + y4 = 0
-1 + y4 = 0
y4 = 1

Row 5:
-y4 + y5 = 0
-1 + y5 = 0
y5 = 1

Therefore

y =
[0]
[1]
[1]
[1]
[1]

Step 2: Solve Ux2 = y

Row 5:
x5 = 1

Row 4:
x4 - 1 = 1
x4 = 2

Row 3:
x3 - 2 = 1
x3 = 3

Row 2:
x2 - 3 = 1
x2 = 4

Row 1:
x1 - 4 = 0
x1 = 4

Therefore
Column 2 =
x2 =
[4]
[4]
[3]
[2]
[1]

--------------------------------------------------
COLUMN 3: Solve Ax3 = e3
--------------------------------------------------

e3 =
[0]
[0]
[1]
[0]
[0]

Step 1: Solve Ly = e3

Row 1:
y1 = 0

Row 2:
y2 = 0

Row 3:
-y2 + y3 = 1
0 + y3 = 1
y3 = 1

Row 4:
-y3 + y4 = 0
-1 + y4 = 0
y4 = 1

Row 5:
-y4 + y5 = 0
-1 + y5 = 0
y5 = 1

Therefore
y =
[0]
[0]
[1]
[1]
[1]

Step 2: Solve Ux3 = y

Row 5:
x5 = 1

Row 4:
x4 = 2

Row 3:
x3 - 2 = 1
x3 = 3

Row 2:
x2 - 3 = 0
x2 = 3

Row 1:
x1 - 3 = 0
x1 = 3

Therefore
Column 3 =

x3 =
[3]
[3]
[3]
[2]
[1]

--------------------------------------------------
Remaining Columns
--------------------------------------------------

By performing the same calculations:

Column 4 =
x4 =
[2]
[2]
[2]
[2]
[1]

Column 5 =
x5 =
[1]
[1]
[1]
[1]
[1]

--------------------------------------------------
Construct the Inverse
--------------------------------------------------

Place the solution vectors as columns:

A^(-1) =
[5 4 3 2 1]
[4 4 3 2 1]
[3 3 3 2 1]
[2 2 2 2 1]
[1 1 1 1 1]


PART (c) - Find A^(-1)

To compute the inverse, solve

Ax = e_i

for each standard basis vector.

For e1:
x1 =
[5]
[4]
[3]
[2]
[1]

For e2:
x2 =
[4]
[4]
[3]
[2]
[1]

For e3:
x3 =
[3]
[3]
[3]
[2]
[1]

For e4:
x4 =
[2]
[2]
[2]
[2]
[1]

For e5:
x5 =
[1]
[1]
[1]
[1]
[1]

Place these vectors as columns of the inverse.

A^(-1) =

[5 4 3 2 1]
[4 4 3 2 1]
[3 3 3 2 1]
[2 2 2 2 1]
[1 1 1 1 1]
"""

'\nPART (c) - Finding A^(-1) by Solving Ax_i = e_i\n\nRecall that the inverse matrix is formed by solving\n\nAx1 = e1\nAx2 = e2\nAx3 = e3\nAx4 = e4\nAx5 = e5\n\nwhere\n\ne1 = [1 0 0 0 0]^T\ne2 = [0 1 0 0 0]^T\ne3 = [0 0 1 0 0]^T\ne4 = [0 0 0 1 0]^T\ne5 = [0 0 0 0 1]^T\n\nThe resulting vectors become the columns of A^(-1).\n\nSince A = LU, solve each system in two steps:\n\n1. Solve Ly = e_i\n2. Solve Ux_i = y\n\n--------------------------------------------------\nCOLUMN 1: Solve Ax1 = e1\n--------------------------------------------------\n\ne1 =\n[1]\n[0]\n[0]\n[0]\n[0]\n\nStep 1: Solve Ly = e1\n\nL =\n[ 1  0  0  0  0 ]\n[-1  1  0  0  0 ]\n[ 0 -1  1  0  0 ]\n[ 0  0 -1  1  0 ]\n[ 0  0  0 -1  1 ]\n\nRow 1:\ny1 = 1\n\nRow 2:\n-y1 + y2 = 0\n-1 + y2 = 0\ny2 = 1\n\nRow 3:\n-y2 + y3 = 0\n-1 + y3 = 0\ny3 = 1\n\nRow 4:\n-y3 + y4 = 0\n-1 + y4 = 0\ny4 = 1\n\nRow 5:\n-y4 + y5 = 0\n-1 + y5 = 0\ny5 = 1\n\nTherefore\n\ny =\n[1]\n[1]\n[1]\n[1]\n[1]\n\nStep 2: Solve Ux1 = y\n\nU =\n[1 -1  0  0  0]\n[0

In [24]:
"""
PART (d) - General SPD Tridiagonal Matrix

Consider an n x n symmetric positive definite tridiagonal matrix

A =
[a1  c1   0    ...      0]
[c1  a2  c2    ...      0]
[0   c2  a3    ...      0]
[...                cn-1]
[0    0  cn-1   an]

Because A is symmetric positive definite, no pivoting is required.

Construct LU as follows.

Set

u1 = a1

For i = 2,...,n:
li = c(i-1)/u(i-1)
ui = ai - li*c(i-1)

Each iteration performs only a constant number of arithmetic operations.
Since there are n iterations, the LU decomposition requires O(n) time.

To solve Ax = b:

1. Solve Ly = b using forward substitution.

   y1 = b1

   yi = bi - li*y(i-1)

2. Solve Ux = y using back substitution.

   xn = yn/un

   xi = (yi - ci*x(i+1))/ui

Both substitutions require O(n) time.
Therefore the entire solution process requires O(n) time.
"""

'\nPART (d) - General SPD Tridiagonal Matrix\n\nConsider an n x n symmetric positive definite tridiagonal matrix\n\nA =\n[a1  c1   0    ...      0]\n[c1  a2  c2    ...      0]\n[0   c2  a3    ...      0]\n[...                cn-1]\n[0    0  cn-1   an]\n\nBecause A is symmetric positive definite, no pivoting is required.\n\nConstruct LU as follows.\n\nSet\n\nu1 = a1\n\nFor i = 2,...,n:\nli = c(i-1)/u(i-1)\nui = ai - li*c(i-1)\n\nEach iteration performs only a constant number of arithmetic operations.\nSince there are n iterations, the LU decomposition requires O(n) time.\n\nTo solve Ax = b:\n\n1. Solve Ly = b using forward substitution.\n\n   y1 = b1\n\n   yi = bi - li*y(i-1)\n\n2. Solve Ux = y using back substitution.\n\n   xn = yn/un\n\n   xi = (yi - ci*x(i+1))/ui\n\nBoth substitutions require O(n) time.\nTherefore the entire solution process requires O(n) time.\n'

In [25]:
"""
PART (d) - Why Computing the Inverse is More Expensive

An n x n inverse matrix contains n^2 entries.
Therefore simply storing or constructing A^(-1) requires at least Ω(n^2) work.

After the inverse is computed, solving

x = A^(-1)b

requires a matrix-vector multiplication.
This multiplication requires Θ(n^2) operations.
Therefore any algorithm based on explicitly computing A^(-1) requires at least Ω(n^2) time.
Since LU decomposition solves the system in O(n) time, the inverse-based approach is asymptotically slower.
"""

'\nPART (d) - Why Computing the Inverse is More Expensive\n\nAn n x n inverse matrix contains n^2 entries.\nTherefore simply storing or constructing A^(-1) requires at least Ω(n^2) work.\n\nAfter the inverse is computed, solving\n\nx = A^(-1)b\n\nrequires a matrix-vector multiplication.\nThis multiplication requires Θ(n^2) operations.\nTherefore any algorithm based on explicitly computing A^(-1) requires at least Ω(n^2) time.\nSince LU decomposition solves the system in O(n) time, the inverse-based approach is asymptotically slower.\n'

In [26]:
"""
PART (e) - General Nonsingular Tridiagonal Matrix Using LUP

Consider

A =
[a1  c1   0    ...      0]
[b2  a2  c2    ...      0]
[0   b3  a3    ...      0]
[...                cn-1]
[0    0  bn    an]

Because A is not necessarily symmetric positive definite, pivoting may be required.
At elimination step i, only rows i and i+1 contain nonzero entries in column i.
Therefore pivot selection only compares two possible pivots.

If necessary, swap the two neighboring rows.
The multiplier is then computed and only a constant number of entries are updated.
Each elimination step therefore requires O(1) work.
Since there are n elimination steps, constructing the LUP decomposition requires O(n) time.

After obtaining
PA = LU

solve
Ly = Pb

using forward substitution and then solve
Ux = y

using back substitution.

Each substitution requires O(n) time.
Therefore solving Ax = b for a nonsingular tridiagonal matrix using LUP decomposition requires O(n) time overall.
"""

'\nPART (e) - General Nonsingular Tridiagonal Matrix Using LUP\n\nConsider\n\nA =\n[a1  c1   0    ...      0]\n[b2  a2  c2    ...      0]\n[0   b3  a3    ...      0]\n[...                cn-1]\n[0    0  bn    an]\n\nBecause A is not necessarily symmetric positive definite, pivoting may be required.\nAt elimination step i, only rows i and i+1 contain nonzero entries in column i.\nTherefore pivot selection only compares two possible pivots.\n\nIf necessary, swap the two neighboring rows.\nThe multiplier is then computed and only a constant number of entries are updated.\nEach elimination step therefore requires O(1) work.\nSince there are n elimination steps, constructing the LUP decomposition requires O(n) time.\n\nAfter obtaining\nPA = LU\n\nsolve\nLy = Pb\n\nusing forward substitution and then solve\nUx = y\n\nusing back substitution.\n\nEach substitution requires O(n) time.\nTherefore solving Ax = b for a nonsingular tridiagonal matrix using LUP decomposition requires O(n) time overa

In [ ]:
"""
Why Splines Are Useful

Suppose we measure temperature at several times:

Time    Temperature
0       20
1       25
2       18
3       30

If someone asks for the temperature at time 1.5, we do not have a direct measurement. We need a way to estimate values between known data points. One approach is linear interpolation, which connects neighboring points with straight lines. Between times 1 and 2, the slope is (18 - 25)/(2 - 1) = -7, giving the estimate T(1.5) = 25 - 7(0.5) = 21.5.

A spline uses smooth cubic curves instead of straight lines. For the interval between times 1 and 2, suppose the temperature values are 25 and 18 and the slopes at the endpoints are D1 = -4 and D2 = 3. We construct a cubic polynomial

f(x) = a + bx + cx^2 + dx^3

where x ranges from 0 to 1 across the interval.

The four conditions are:
1. f(0) = 25
2. f(1) = 18
3. f'(0) = -4
4. f'(1) = 3

From f(0) = 25 we obtain a = 25.

The derivative is
f'(x) = b + 2cx + 3dx^2.

Using f'(0) = -4 gives b = -4.

Using f(1) = 18 gives
25 - 4 + c + d = 18

which simplifies to
c + d = -3.

Using f'(1) = 3 gives
-4 + 2c + 3d = 3

which simplifies to
2c + 3d = 7.

Solving the system
c + d = -3
2c + 3d = 7

gives
d = 13
c = -16.

Therefore the cubic spline is
f(x) = 25 - 4x - 16x^2 + 13x^3.

To estimate the temperature at time 1.5, use x = 0.5:
f(0.5) = 25 - 4(0.5) - 16(0.5)^2 + 13(0.5)^3

= 25 - 2 - 4 + 1.625
= 20.625.

Thus the spline estimate for the temperature at time 1.5 is approximately 20.625 degrees.
Splines are useful because they create smooth curves through known data points and often
model physical systems more realistically than straight-line interpolation.
"""